In [1]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
from peft import PeftModel


/home/avinash/.pyenv/versions/llm-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen3-4B-Base",
    trust_remote_code=True,
)

# =====================================================
# 4-BIT QUANTIZATION
# =====================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

# =====================================================
# LOAD BASE MODEL
# =====================================================

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    "/home/avinash/Projects/Custom-LN-Translator-Training/Model-Checkpoints/merged_qwen3_ln_translator",
    quantization_config=bnb_config,
    device_map="cuda",
    trust_remote_code=True,
)

Loading base model...


/home/avinash/.pyenv/versions/llm-env/lib/python3.10/site-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [7]:

# =====================================================
# TEST PASSAGE
# =====================================================

jp_text = """

冷たい微風が頬を撫で、冷え切った体が身震いした。頬に当たる硬い感触と下半身の刺すような冷たい感触に「うっ」と呻き声を上げてハジメは目を覚ました。

ボーとする頭、ズキズキと痛む全身に眉根を寄せながら両腕に力を入れて上体を起こす。

「痛っ~、ここは......僕は確か......」

ふらつく頭を片手で押さえながら、記憶を辿りつつ辺りを見回す。

周りは薄暗いが緑光石の発光のおかげで何も見えないほどではない。視線の先には幅五メートル程の川があり、ハジメの下半身が浸かっていた。上半身が、突き出た川辺の岩に引っかかって乗り上げたようだ。

「そうだ......確か、橋が壊れて落ちたんだ。......それで......」

霧がかかったようだった頭が回転を始める。

ハジメが奈落に落ちていながら助かったのは全くの幸運だった。


「グルゥア!!」
"""

prompt = f"""Translate the following Japanese light novel passage into natural English light novel prose.

Preserve meaning, tone, dialogue, names, honorific intent, and paragraph breaks.
Output only the English translation.

Japanese:
{jp_text}

English Translation:
"""

# =====================================================
# TOKENIZER
# =====================================================

In [8]:
model=base_model.to("cuda")
model.eval()

# =====================================================
# TOKENIZE
# =====================================================

inputs = tokenizer(
    prompt,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) for k, v in inputs.items()}

# =====================================================
# GENERATE
# =====================================================

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=4096,
        do_sample=False,
        temperature=0.0,
    )

# =====================================================
# DECODE
# =====================================================

translation = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True,
)

print("\n=== OUTPUT ===\n")
print(translation)


=== OUTPUT ===

Translate the following Japanese light novel passage into natural English light novel prose.

Preserve meaning, tone, dialogue, names, honorific intent, and paragraph breaks.
Output only the English translation.

Japanese:


冷たい微風が頬を撫で、冷え切った体が身震いした。頬に当たる硬い感触と下半身の刺すような冷たい感触に「うっ」と呻き声を上げてハジメは目を覚ました。

ボーとする頭、ズキズキと痛む全身に眉根を寄せながら両腕に力を入れて上体を起こす。

「痛っ~、ここは......僕は確か......」

ふらつく頭を片手で押さえながら、記憶を辿りつつ辺りを見回す。

周りは薄暗いが緑光石の発光のおかげで何も見えないほどではない。視線の先には幅五メートル程の川があり、ハジメの下半身が浸かっていた。上半身が、突き出た川辺の岩に引っかかって乗り上げたようだ。

「そうだ......確か、橋が壊れて落ちたんだ。......それで......」

霧がかかったようだった頭が回転を始める。

ハジメが奈落に落ちていながら助かったのは全くの幸運だった。


「グルゥア!!」


English Translation:
Hajime opened his eyes.

"Ugh~! This is......I'm sure......"

He pressed his head with one hand while looking around.

The surroundings were dim, but thanks to the green light stone, nothing could be seen. In front of his line of sight was a river about five meters wide, and Hajime's lower half was submerged in it. His upper half was caught by a rock protru